# Análise de Chamados — Projeto Final

Importação dos dados da pasta `dados` e geração de insights primários sobre a operação de atendimento ao cliente: volume de chamados, taxa de resolução, tempos de espera e atendimento, satisfação, custos e distribuições por canal, região e motivo.

In [0]:
# ============================================================
# Importação dos dados da pasta "dados"
# ============================================================

import os

base_path = "/Workspace/Users/email/case-final-rocket-lab/dados"

# Lista de arquivos disponíveis
arquivos = sorted(os.listdir(base_path))
print("Arquivos encontrados:")
for a in arquivos:
    fp = os.path.join(base_path, a)
    size_kb = os.path.getsize(fp) / 1024
    print(f"  {a}  ({size_kb:.0f} KB)")

Arquivos encontrados:
  Chamados_Hora.CSV  (11277 KB)
  base_atendentes.csv  (1 KB)
  base_motivos.csv  (1 KB)
  canais.csv  (0 KB)
  chamados.csv  (14478 KB)
  clientes.csv  (4461 KB)
  custos.csv  (3265 KB)
  pesquisa_satisfacao.csv  (1666 KB)


In [0]:
# ============================================================
# Carregar todos os CSVs em DataFrames Spark
# ============================================================
# Os arquivos usam vírgula como separador decimal e aspas duplas.
# Chamados_Hora.CSV não tem aspas; os demais possuem.

data_path = f"file://{base_path}"

# --- chamados.csv ---
df_chamados = spark.read.csv(
    f"{data_path}/chamados.csv",
    header=True,
    inferSchema=True,
    quote='"',
    sep=",",
    encoding="UTF-8"
)

# --- Chamados_Hora.CSV (sem aspas, encoding latin1 para "às") ---
df_chamados_hora = spark.read.csv(
    f"{data_path}/Chamados_Hora.CSV",
    header=True,
    inferSchema=True,
    sep=",",
    encoding="ISO-8859-1"
)

# --- clientes.csv ---
df_clientes = spark.read.csv(
    f"{data_path}/clientes.csv",
    header=True,
    inferSchema=True,
    quote='"',
    sep=",",
    encoding="UTF-8"
)

# --- custos.csv ---
df_custos = spark.read.csv(
    f"{data_path}/custos.csv",
    header=True,
    inferSchema=True,
    quote='"',
    sep=",",
    encoding="UTF-8"
)

# --- pesquisa_satisfacao.csv ---
df_pesquisa = spark.read.csv(
    f"{data_path}/pesquisa_satisfacao.csv",
    header=True,
    inferSchema=True,
    quote='"',
    sep=",",
    encoding="UTF-8"
)

# --- canais.csv ---
df_canais = spark.read.csv(
    f"{data_path}/canais.csv",
    header=True,
    inferSchema=True,
    quote='"',
    sep=",",
    encoding="UTF-8"
)

# --- base_motivos.csv ---
df_motivos = spark.read.csv(
    f"{data_path}/base_motivos.csv",
    header=True,
    inferSchema=True,
    quote='"',
    sep=",",
    encoding="UTF-8"
)

# --- base_atendentes.csv ---
df_atendentes = spark.read.csv(
    f"{data_path}/base_atendentes.csv",
    header=True,
    inferSchema=True,
    quote='"',
    sep=",",
    encoding="UTF-8"
)

# Dicionário para facilitar o acesso
tabelas = {
    "chamados": df_chamados,
    "chamados_hora": df_chamados_hora,
    "clientes": df_clientes,
    "custos": df_custos,
    "pesquisa_satisfacao": df_pesquisa,
    "canais": df_canais,
    "base_motivos": df_motivos,
    "base_atendentes": df_atendentes,
}

print("✅ Todos os arquivos carregados com sucesso!")
print(f"Total de tabelas: {len(tabelas)}")

✅ Todos os arquivos carregados com sucesso!
Total de tabelas: 8


In [0]:
# ============================================================
# Visão geral: volumes e schemas de cada tabela
# ============================================================

print("=" * 60)
print("RESUMO DE VOLUMES")
print("=" * 60)
for nome, df in tabelas.items():
    qtd = df.count()
    cols = len(df.columns)
    print(f"  {nome:25s} → {qtd:>8,} linhas | {cols} colunas")
print("=" * 60)

RESUMO DE VOLUMES
  chamados                  →  134,474 linhas | 11 colunas
  chamados_hora             →  131,838 linhas | 5 colunas
  clientes                  →   47,705 linhas | 7 colunas
  custos                    →  131,069 linhas | 3 colunas
  pesquisa_satisfacao       →   81,021 linhas | 3 colunas
  canais                    →        4 linhas | 4 colunas
  base_motivos              →       14 linhas | 4 colunas
  base_atendentes           →       25 linhas | 6 colunas


In [0]:
# ============================================================
# Amostras das tabelas principais
# ============================================================

print("--- CHAMADOS (5 linhas) ---")
display(df_chamados.limit(5))
print("\n--- CLIENTES (5 linhas) ---")
display(df_clientes.limit(5))
print("\n--- CUSTOS (5 linhas) ---")
display(df_custos.limit(5))
print("\n--- PESQUISA SATISFACAO (5 linhas) ---")
display(df_pesquisa.limit(5))
print("\n--- CANAIS ---")
display(df_canais)
print("\n--- BASE MOTIVOS ---")
display(df_motivos)
print("\n--- BASE ATENDENTES ---")
display(df_atendentes)

--- CHAMADOS (5 linhas) ---


id_chamado,id_cliente,motivo,canal,resolvido,hora_abertura_chamado,hora_inicio_atendimento,hora_finalizacao_atendimento,tempo_espera,tempo_atendimento,id_atendente
76696,74719850215,Renegociação de dívida,atendimento inicial,Sim,null,null,null,49,441,4
44377,92983951302,CONSULTA DE CONTRATO,Atendimento_Inicial,Sim,null,null,null,374,355,8
34306,77740207578,Consulta de limite,ATENDIMENTO INICIAL,Sim,null,null,null,110,985,6
52060,36276971799,Renegociação de dívida,Ura,Não,null,igual a hora de abertura,null,NULL,216,9
25271,53061261807,Renegociação de dívida,Atendimento_Inicial,N,null,null,null,96,384,2



--- CLIENTES (5 linhas) ---


id_cliente,nome,email,regiao,idade,data_cadastro,dia_vencimento_fatura
56324416706,S�rgio Martins Miranda,sergio.miranda98@hotmail.com,Sul,51,14/10/2024,10
97653229900,Leandro Magalh�es Lopes,lopes.leandro@yahoo.com.br,Sudeste,51,2026-07-07,10
15132904850,Giovana Soares Vasconcelos,giovana.vasconcelos26@hotmail.com,Sudeste,52,16/02/2019,25
62566924227,Adriana Nascimento,adriana96@gmail.com,Nordeste,39,27/08/2025,20
35213062014,Vera Duarte,vera.duarte@ig.com.br,Sudeste,34,07/05/2026,15



--- CUSTOS (5 linhas) ---


...

id_canal,nome_canal,tipo_canal,status_canal
1,URA,Autosservico,ativo
2,CHATBOT,Autosservico,ativo
3,ATENDIMENTO INICIAL,Humano,ativo
4,ATENDIMENTO ESPECIALIZADO,Humano,ativo



--- BASE MOTIVOS ---


id_motivo,nome_motivo,categoria,criticidade
11,Bloqueio de cartão,Cartao,Alta
7,Compra não autorizada,Cartao,alta
13,Dúvidas gerais sobre programa de pontos,Beneficios,baixa
14,Transferência de agência,null,Media
9,Problema com aplicativo,Atendimento,Media
8,Consulta de contrato,Financeiro,Media
15,Renegociação de dívida,Financeiro,alta
16,Cancelamento de conta,null,Alta
4,"Alteração de dados cadastrais (vencimento da fatura, telefone, email)",Cadastral,baixa
3,Contestação de fatura,Financeiro,alta



--- BASE ATENDENTES ---


id_atendente,nome_atendente,nivel_atendimento,data_admissao,data_desligamento,status_atendente
1,Marcela França,1,2019-04-18,2025-11-16,inativo
2,Zilda Esteves,1,2023-09-12,2026-01-24,inativo
3,William Marques,1,2019-05-28,2025-04-15,inativo
4,Solange Silveira,Nível 1,2019-10-07,null,ativo
5,Joaquim Barros,Nível 1,2020-12-06,2026-05-23,inativo
6,Roberta Assunção,1,2021-10-29,null,ativo
7,Luiza Galvão,Nível 1,2022-04-24,null,ativo
8,Josefa Brito,Nível 1,2023-01-25,null,ativo
9,Jaqueline Campos,1,2019-04-19,null,ativo
10,Marcos Torres,1,2020-01-20,null,ativo


In [0]:
# ============================================================
# Verificação de qualidade dos dados
# ============================================================
from pyspark.sql.functions import col, count, when, isnan, isnull, trim, upper, lower, regexp_replace, regexp_extract

print("=" * 70)
print("1. CONTAGEM DE NULOS POR TABELA")
print("=" * 70)
for nome, df in tabelas.items():
    null_exprs = []
    for c in df.columns:
        field = df.schema[c]
        if field.dataType.typeName() == "string":
            null_exprs.append(count(when(isnull(col(c)) | (col(c) == ""), c)).alias(c))
        else:
            null_exprs.append(count(when(isnull(col(c)), c)).alias(c))
    null_counts = df.select(null_exprs).collect()[0]
    total = df.count()
    print(f"\n  [{nome}] ({total:,} linhas)")
    for c in df.columns:
        nc = null_counts[c]
        pct = (nc / total * 100) if total > 0 else 0
        flag = " ⚠️" if pct > 5 else ""
        if nc > 0:
            print(f"    {c:35s}: {nc:>7,} nulos ({pct:.1f}%){flag}")

print("\n" + "=" * 70)
print("2. INCONSISTÊNCIAS DETECTADAS")
print("=" * 70)

# 2a. chamados.resolvido — valores distintos
print("\n  [chamados.resolvido] valores distintos:")
df_chamados.select("resolvido").distinct().show()

# 2b. chamados.canal — valores distintos (case inconsistency)
print("  [chamados.canal] valores distintos (raw):")
df_chamados.select("canal").distinct().show(truncate=False)
print("  [chamados.canal] valores distintos (normalizado = upper+trim):")
df_chamados.select(upper(trim(col("canal"))).alias("canal_norm")).distinct().show(truncate=False)

# 2c. chamados.tempo_espera — contém "NULL" como string
null_str_espera = df_chamados.filter(col("tempo_espera") == "NULL").count()
print(f"  [chamados.tempo_espera] linhas com 'NULL' (string): {null_str_espera:,}")

null_str_atend = df_chamados.filter(col("tempo_atendimento") == "NULL").count()
print(f"  [chamados.tempo_atendimento] linhas com 'NULL' (string): {null_str_atend:,}")

# 2d. custos.custo — formatos distintos
print("  [custos.custo] amostra de formatos:")
df_custos.select("custo").distinct().limit(10).show(truncate=False)

# 2e. base_motivos.criticidade — case inconsistency
print("  [base_motivos.criticidade] valores distintos:")
df_motivos.select("criticidade").distinct().show()

# 2f. base_atendentes.nivel_atendimento — valores inconsistentes
print("  [base_atendentes.nivel_atendimento] valores distintos:")
df_atendentes.select("nivel_atendimento").distinct().show()

# 2g. Duplicatas em chamados por id_chamado
dup_chamados = df_chamados.groupBy("id_chamado").count().filter("count > 1").count()
print(f"  [chamados] id_chamado duplicados: {dup_chamados:,}")

# 2h. Duplicatas em clientes por id_cliente
dup_clientes = df_clientes.groupBy("id_cliente").count().filter("count > 1").count()
print(f"  [clientes] id_cliente duplicados: {dup_clientes:,}")

1. CONTAGEM DE NULOS POR TABELA

  [chamados] (134,474 linhas)
    resolvido                          :   2,063 nulos (1.5%)
    hora_abertura_chamado              : 134,474 nulos (100.0%) ⚠️
    hora_inicio_atendimento            :  57,877 nulos (43.0%) ⚠️
    hora_finalizacao_atendimento       : 134,474 nulos (100.0%) ⚠️

  [chamados_hora] (131,838 linhas)
    Hora_Inicio_Atendimento            :   3,414 nulos (2.6%)

  [clientes] (47,705 linhas)
    idade                              :     459 nulos (1.0%)

  [custos] (131,069 linhas)

  [pesquisa_satisfacao] (81,021 linhas)
    nota_atendimento                   :   2,842 nulos (3.5%)

  [canais] (4 linhas)

  [base_motivos] (14 linhas)
    categoria                          :       3 nulos (21.4%) ⚠️

  [base_atendentes] (25 linhas)
    data_desligamento                  :      20 nulos (80.0%) ⚠️

2. INCONSISTÊNCIAS DETECTADAS

  [chamados.resolvido] valores distintos:
+-------------+
|    resolvido|
+-------------+
|            

In [0]:
# ============================================================
# Limpeza e normalização dos dados
# ============================================================
from pyspark.sql.functions import (
    col, count, when, isnan, isnull, trim, upper, lower,
    regexp_replace, regexp_extract, lit, coalesce, round as spark_round,
    avg, sum as spark_sum, desc, asc, split, size, expr, to_date
)

# --- 1. Normalizar chamados ---
df_chamados_clean = df_chamados \
    .withColumn("canal_norm",
        regexp_replace(
            regexp_replace(
                upper(trim(col("canal"))),
                "_", " "
            ),
            "ATEND\\.", "ATENDIMENTO "
        )
    ) \
    .withColumn("canal_norm",
        when(col("canal_norm") == "U.R.A", "URA")
        .when(col("canal_norm") == "CHAT BOT", "CHATBOT")
        .when(col("canal_norm") == "ATENDIMENTO  INICIAL", "ATENDIMENTO INICIAL")
        .when(col("canal_norm") == "ATENDIMENTO  ESPECIALIZADO", "ATENDIMENTO ESPECIALIZADO")
        .otherwise(col("canal_norm"))
    ) \
    .withColumn("resolvido_norm",
        when(upper(trim(col("resolvido"))).isin("SIM", "S", "1"), "Sim")
        .when(upper(trim(col("resolvido"))).isin("NÃO", "NAO", "NAO RESOLVIDO", "N", "0"), "Não")
 .otherwise(None)
    ) \
    .withColumn("tempo_espera_num",
        when(col("tempo_espera") == "NULL", None)
        .otherwise(col("tempo_espera").cast("double"))
    ) \
    .withColumn("tempo_atendimento_num",
        when(col("tempo_atendimento") == "NULL", None)
        .otherwise(col("tempo_atendimento").cast("double"))
    ) \
    .withColumn("motivo_norm", upper(trim(col("motivo"))))

# --- 2. Limpar custos ---
df_custos_clean = df_custos \
    .withColumn("custo_num",
        regexp_replace(
            regexp_replace(
            regexp_replace(
                col("custo"),
                "R\\$\\s*", ""  # remove R$
            ),
            "reais", ""  # remove 'reais'
        ),
        "\\.", ""  # remove ponto de milhar (se houver)
    )
    ) \
    .withColumn("custo_num", regexp_replace(col("custo_num"), ",", ".").cast("double"))

# --- 3. Limpar pesquisa_satisfacao ---
df_pesquisa_clean = df_pesquisa \
    .withColumn("nota_num",
        when(col("nota_atendimento") == "NULL", None)
        .otherwise(col("nota_atendimento").cast("double"))
    )

# --- 4. Normalizar base_motivos ---
df_motivos_clean = df_motivos \
    .withColumn("criticidade_norm", upper(trim(col("criticidade")))) \
    .withColumn("categoria_norm", upper(trim(col("categoria"))))

# --- 5. Normalizar base_atendentes ---
df_atendentes_clean = df_atendentes \
    .withColumn("nivel_norm",
        when(upper(trim(col("nivel_atendimento"))).isin("1", "NÍVEL 1"), "Nivel 1")
        .when(upper(trim(col("nivel_atendimento"))).isin("2", "NÍVEL 2", "N2"), "Nivel 2")
        .otherwise(upper(trim(col("nivel_atendimento"))))
    )

print("✅ Limpeza e normalização concluídas!")

✅ Limpeza e normalização concluídas!


In [0]:
# ============================================================
# INSIGHTS PRIMÁRIOS
# ============================================================

print("=" * 70)
print("📊 INSIGHT 1: PANORAMA GERAL DA OPERAÇÃO")
print("=" * 70)

total_chamados = df_chamados_clean.count()
total_clientes = df_clientes.count()
total_atendentes = df_atendentes_clean.filter(col("status_atendente") == "ativo").count()
total_custo = df_custos_clean.agg(spark_sum("custo_num")).collect()[0][0] or 0

df_resol = df_chamados_clean.filter(col("resolvido_norm").isNotNull())
total_resol = df_resol.count()
resol_sim = df_resol.filter(col("resolvido_norm") == "Sim").count()
taxa_resol = (resol_sim / total_resol * 100) if total_resol > 0 else 0

print(f"  Total de chamados:       {total_chamados:>10,}")
print(f"  Total de clientes:       {total_clientes:>10,}")
print(f"  Atendentes ativos:       {total_atendentes:>10}")
print(f"  Custo total (R$):        {total_custo:>10,.2f}")
print(f"  Taxa de resolução:       {taxa_resol:>9.1f}%  ({resol_sim:,} de {total_resol:,})")
print()

df_espera = df_chamados_clean.filter(col("tempo_espera_num").isNotNull())
df_atend = df_chamados_clean.filter(col("tempo_atendimento_num").isNotNull())
avg_espera = df_espera.agg(avg("tempo_espera_num")).collect()[0][0] or 0
med_espera = df_espera.approxQuantile("tempo_espera_num", [0.5], 0.01)[0]
avg_atend = df_atend.agg(avg("tempo_atendimento_num")).collect()[0][0] or 0
med_atend = df_atend.approxQuantile("tempo_atendimento_num", [0.5], 0.01)[0]

print(f"  Tempo médio de espera:   {avg_espera:>9.1f} min (mediana: {med_espera:.1f})")
print(f"  Tempo médio de atend.:   {avg_atend:>9.1f} min (mediana: {med_atend:.1f})")
print()

df_nota = df_pesquisa_clean.filter(col("nota_num").isNotNull())
avg_nota = df_nota.agg(avg("nota_num")).collect()[0][0] or 0
total_pesq = df_nota.count()
print(f"  Satisfação média:        {avg_nota:>9.2f}/5  ({total_pesq:,} respostas)")

📊 INSIGHT 1: PANORAMA GERAL DA OPERAÇÃO
  Total de chamados:          134,474
  Total de clientes:           47,705
  Atendentes ativos:               20
  Custo total (R$):        42,239,199.09
  Taxa de resolução:            50.9%  (67,402 de 132,411)

  Tempo médio de espera:       344.2 min (mediana: 298.0)
  Tempo médio de atend.:       216.7 min (mediana: 133.0)

  Satisfação média:             3.94/5  (75,343 respostas)


In [0]:
# ============================================================
print("=" * 70)
print("📊 INSIGHT 2: CHAMADOS POR CANAL")
print("=" * 70)

df_canal_agg = df_chamados_clean \
    .filter(col("canal_norm").isNotNull()) \
    .groupBy("canal_norm") \
    .agg(
        count("*").alias("qtd_chamados"),
        spark_round(count(when(col("resolvido_norm") == "Sim", True)) / count(when(col("resolvido_norm").isNotNull(), True)) * 100, 1).alias("taxa_resolucao_pct"),
        spark_round(avg("tempo_espera_num"), 1).alias("espera_media_min"),
        spark_round(avg("tempo_atendimento_num"), 1).alias("atend_media_min")
    ) \
    .orderBy(desc("qtd_chamados"))

display(df_canal_agg)

# ============================================================
print("=" * 70)
print("📊 INSIGHT 3: TOP 10 MOTIVOS DE CHAMADO")
print("=" * 70)

df_motivo_agg = df_chamados_clean \
    .filter(col("motivo_norm").isNotNull()) \
    .groupBy("motivo_norm") \
    .agg(
        count("*").alias("qtd_chamados"),
        spark_round(count(when(col("resolvido_norm") == "Sim", True)) / count(when(col("resolvido_norm").isNotNull(), True)) * 100, 1).alias("taxa_resolucao_pct"),
        spark_round(avg("tempo_espera_num"), 1).alias("espera_media_min")
    ) \
    .orderBy(desc("qtd_chamados")) \
    .limit(10)

display(df_motivo_agg)

# ============================================================
print("=" * 70)
print("📊 INSIGHT 4: CHAMADOS POR REGIÃO")
print("=" * 70)

df_regiao = df_chamados_clean \
    .join(df_clientes, on="id_cliente", how="left") \
    .filter(col("regiao").isNotNull()) \
    .groupBy("regiao") \
    .agg(
        count("*").alias("qtd_chamados"),
        spark_round(avg("idade"), 1).alias("idade_media"),
        spark_round(count(when(col("resolvido_norm") == "Sim", True)) / count(when(col("resolvido_norm").isNotNull(), True)) * 100, 1).alias("taxa_resolucao_pct"),
        spark_round(avg("tempo_espera_num"), 1).alias("espera_media_min")
    ) \
    .orderBy(desc("qtd_chamados"))

display(df_regiao)

# ============================================================
print("=" * 70)
print("📊 INSIGHT 5: SATISFAÇÃO E CUSTO POR CANAL")
print("=" * 70)

df_sat_canal = df_chamados_clean \
    .join(df_pesquisa_clean, on="id_chamado", how="left") \
    .join(df_custos_clean, on="id_chamado", how="left") \
    .filter(col("canal_norm").isNotNull()) \
    .groupBy("canal_norm") \
    .agg(
        spark_round(avg("nota_num"), 2).alias("satisfacao_media"),
        count(when(col("nota_num").isNotNull(), True)).alias("qtd_pesquisas"),
        spark_round(spark_sum("custo_num"), 2).alias("custo_total"),
        spark_round(avg("custo_num"), 2).alias("custo_medio")
    ) \
    .orderBy(desc("custo_total"))

display(df_sat_canal)

📊 INSIGHT 2: CHAMADOS POR CANAL


canal_norm,qtd_chamados,taxa_resolucao_pct,espera_media_min,atend_media_min
ATENDIMENTO INICIAL,40486,52.5,346.1,366.3
URA,40369,45.5,null,210.5
CHATBOT,36228,49.0,null,113.6
ATENDIMENTO ESPECIALIZADO,17391,63.8,340.0,101.6


📊 INSIGHT 3: TOP 10 MOTIVOS DE CHAMADO


motivo_norm,qtd_chamados,taxa_resolucao_pct,espera_media_min
CONSULTA DE FATURA,17025,67.2,347.4
PROBLEMA COM APLICATIVO,14598,37.8,348.7
CONSULTA DE LIMITE,14196,65.7,342.5
BLOQUEIO DE CARTÃO,12974,63.5,340.0
COMPRA NÃO AUTORIZADA,12877,32.4,347.8
RENEGOCIAÇÃO DE DÍVIDA,12214,30.8,344.3
CONTESTAÇÃO DE FATURA,11043,36.7,344.7
DESBLOQUEIO DE CARTÃO,8725,63.1,338.8
CANCELAMENTO DE CONTA,6029,33.4,339.8
CONSULTA DE CONTRATO,5668,59.6,346.3


📊 INSIGHT 4: CHAMADOS POR REGIÃO


regiao,qtd_chamados,idade_media,taxa_resolucao_pct,espera_media_min
Sudeste,50483,53.9,50.8,343.7
Nordeste,31483,54.3,51.0,343.9
Sul,17827,53.8,51.1,342.8
Centro-Oeste,10220,54.5,51.4,353.4
Norte,8384,53.2,51.5,341.6
SUDESTE,4799,54.1,51.0,335.3
NORDESTE,2611,53.9,49.6,350.1
Sudeste,1764,54.1,50.7,342.3
SUL,1690,56.2,49.7,338.5
Nordeste,1190,53.2,49.3,346.2


📊 INSIGHT 5: SATISFAÇÃO E CUSTO POR CANAL


canal_norm,satisfacao_media,qtd_pesquisas,custo_total,custo_medio
ATENDIMENTO ESPECIALIZADO,2.84,9506,2.449346442E7,1411.16
ATENDIMENTO INICIAL,3.67,23055,1.59784283E7,394.5
URA,4.28,23860,2105097.73,52.13
CHATBOT,4.36,20881,607216.74,16.75


In [0]:
# ============================================================
print("=" * 70)
print("📊 INSIGHT 6: TOP 10 ATENDENTES POR VOLUME E RESOLUÇÃO")
print("=" * 70)

df_atend_agg = df_chamados_clean \
    .filter(col("id_atendente").isNotNull() & col("resolvido_norm").isNotNull()) \
    .join(df_atendentes_clean, on="id_atendente", how="left") \
    .groupBy("id_atendente", "nome_atendente", "nivel_norm") \
    .agg(
        count("*").alias("qtd_chamados"),
        spark_round(count(when(col("resolvido_norm") == "Sim", True)) / count("*") * 100, 1).alias("taxa_resolucao_pct"),
        spark_round(avg("tempo_atendimento_num"), 1).alias("atend_media_min"),
        spark_round(avg("tempo_espera_num"), 1).alias("espera_media_min")
    ) \
    .orderBy(desc("qtd_chamados")) \
    .limit(10)

display(df_atend_agg)

# ============================================================
print("=" * 70)
print("📊 INSIGHT 7: DISTRIBUIÇÃO DE SATISFAÇÃO")
print("=" * 70)

df_nota_dist = df_pesquisa_clean \
    .filter(col("nota_num").isNotNull()) \
    .groupBy("nota_num") \
    .agg(count("*").alias("qtd_respostas")) \
    .orderBy("nota_num")

display(df_nota_dist)

# ============================================================
print("=" * 70)
print("📊 INSIGHT 8: CUSTO TOTAL POR REGIÃO")
print("=" * 70)

df_custo_regiao = df_chamados_clean \
    .join(df_custos_clean, on="id_chamado", how="left") \
    .join(df_clientes, on="id_cliente", how="left") \
    .filter(col("regiao").isNotNull() & col("custo_num").isNotNull()) \
    .withColumn("regiao_norm", upper(trim(col("regiao")))) \
    .groupBy("regiao_norm") \
    .agg(
        spark_round(spark_sum("custo_num"), 2).alias("custo_total"),
        spark_round(avg("custo_num"), 2).alias("custo_medio"),
        count("*").alias("qtd_chamados")
    ) \
    .orderBy(desc("custo_total"))

display(df_custo_regiao)

📊 INSIGHT 6: TOP 10 ATENDENTES POR VOLUME E RESOLUÇÃO


id_atendente,nome_atendente,nivel_norm,qtd_chamados,taxa_resolucao_pct,atend_media_min,espera_media_min
13,Daniel Fonseca,Nivel 1,6087,47.5,263.3,352.0
12,Marilene Nascimento,Nivel 1,6081,46.4,242.9,347.8
6,Roberta Assunção,Nivel 1,6048,49.3,238.0,343.2
4,Solange Silveira,Nivel 1,6036,52.1,259.2,342.2
8,Josefa Brito,Nivel 1,6002,51.6,304.8,345.5
11,Maria Eduarda Freitas,Nivel 1,5998,50.7,276.1,350.2
7,Luiza Galvão,Nivel 1,5965,52.2,283.1,344.5
9,Jaqueline Campos,Nivel 1,5932,53.1,283.0,347.0
21,Eduardo Lacerda,Nivel 1,5846,45.9,210.7,362.3
10,Marcos Torres,Nivel 1,5808,51.8,287.7,347.8


📊 INSIGHT 7: DISTRIBUIÇÃO DE SATISFAÇÃO


nota_num,qtd_respostas
-1.0,421
0.0,393
1.0,2764
2.0,4814
3.0,13498
4.0,23327
5.0,29719
6.0,407


📊 INSIGHT 8: CUSTO TOTAL POR REGIÃO


regiao_norm,custo_total,custo_medio,qtd_chamados
SUDESTE,1.832355208E7,322.94,56740
NORDESTE,1.15291438E7,328.6,35086
SUL,6468018.15,324.29,19945
CENTRO-OESTE,3472931.86,306.31,11338
NORTE,2751472.98,297.94,9235


## Resumo dos Insights Primários

### Panorama Geral
- **134.474 chamados** atendidos por **20 atendentes ativos**, para **47.705 clientes**
- **Taxa de resolução: 50,9%** — apenas metade dos chamados são resolvidos
- **Custo total: R$ 42,2 milhões**
- **Satisfação média: 3,94/5** (75.343 respostas de pesquisa)
- Tempo médio de espera: **344 min** (mediana: 298 min) — elevado

### Por Canal
| Canal | Volume | Resolução | Espera (min) | Atend. (min) |
| --- | --- | --- | --- | --- |
| Atendimento Inicial | 40.486 | 52,5% | 346 | 366 |
| URA | 40.369 | 45,5% | — | 211 |
| Chatbot | 36.228 | 49,0% | — | 114 |
| Atend. Especializado | 17.391 | 63,8% | 340 | 102 |

- Atendimento Especializado tem a **melhor resolução (63,8%)** mas a **pior satisfação (2,84)**
- Chatbot é o mais rápido (114 min) e com boa satisfação (4,36)

### Top Motivos
- **Consulta de fatura** (17.025) e **Problema com aplicativo** (14.598) lideram em volume
- **Renegociação de dívida** tem a pior taxa de resolução (30,8%)
- **Compra não autorizada** também tem baixa resolução (32,4%)

### Custo e Satisfação por Canal
- Atendimento Especializado: custo médio **R$ 1.411/chamado** (vs R$ 52 da URA)
- Sudeste concentra **43% do custo total** (R$ 18,3M)

### Pontos de Atenção (Qualidade dos Dados)
- `resolvido` com 8 valores distintos (Sim, Não, S, N, 0, 1, NULL, "Nao resolvido") — normalizado
- `canal` com 19 variações de case/spacing — normalizado para 4 valores
- `custo` em 3 formatos ("0,39", "R$ 1,46", "11.14reais") — limpo
- `tempo_espera` com 77.149 "NULL" como string (57,4%)
- 2.636 `id_chamado` duplicados
- Notas de satisfação contêm valores -1, 0 e 6 (fora do range 1-5)
- `regiao` com inconsistência de case/whitespace ("Sudeste" vs "SUDESTE" vs " Sudeste")